# Section 4: Non-Gaussian and bounded filters — QCEFF

*(Replaces DART_LAB slide deck Section 4.)*

Many geophysical quantities are **bounded** (concentrations, rain rates,
sea ice fraction) or strongly **skewed**. Fitting a Gaussian to a prior of
non-negative values puts probability below zero — and an EAKF can then
produce *negative posterior members*. This section introduces the
**Quantile-Conserving Ensemble Filter Framework (QCEFF)**, which lets the
filter use any continuous distribution.

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import pydartlab as dl
import pydartlab.apps as apps

## The rank histogram filter, revisited

Section 1 introduced the RHF: represent the prior with $1/(N{+}1)$
probability between adjacent sorted members (Gaussian tails outside),
multiply by the likelihood bin-by-bin, and place posterior members at
constant quantiles. Because it never assumes the prior is Gaussian, it is
the gateway to non-Gaussian assimilation.

## Exercise: RHF priors and posteriors

In `oned_ensemble`, select **RHF** and update strongly non-Gaussian
ensembles (bimodal, skewed, with outliers). The green/blue curves are now
the actual rank-histogram prior/posterior — not Gaussians. Also try RHF
in `oned_cycle`; both stay available in every later tool.

In [ ]:
oe = apps.oned_ensemble()
oe.filter_radio.value = "RHF"
oe.set_ensemble([0.2, 0.5, 0.7, 0.9, 3.5])  # skewed with an outlier
oe.update_ensemble()
oe

## Exercise: `bounded_oned_ensemble`

A non-negative variable, three filters:

* **EAKF** knows nothing about the bound — *challenge: create a prior and
  observation that produce negative posterior members* (hint: prior
  bunched near zero, observation with large error SD).
* **Gamma**: fits a gamma distribution to the prior and uses a gamma
  likelihood; the posterior is an analytic gamma. Bound respected.
* **BNRHF**: the rank histogram filter with its left tail bounded at zero.
  Bound respected, no parametric assumption.

In [ ]:
bo = apps.bounded_oned_ensemble()
bo.set_ensemble([0.1, 0.3, 0.5, 0.8, 1.2])
bo.update_ensemble()  # EAKF first: look for negative members
bo

In [ ]:
# Compare all three filters on the same prior, scripted:
ens = np.array([0.1, 0.3, 0.5, 0.8, 1.2])
for f in ("EAKF", "Gamma", "BNRHF"):
    bo.filter_radio.value = f
    bo.update_ensemble()
    post = np.round(np.sort(bo.last_posterior), 3)
    print(f"{f:6s} posterior: {post}  min = {post.min()}")

## The quantile-conserving framework

The general recipe behind both bounded filters:

1. Choose a continuous prior distribution and get its CDF $F_p$.
2. Compute each member's **quantile**: $q_n = F_p(x_n)$.
3. Update the *distribution* (filter, inflate, ...) to get the analysis
   CDF $F_a$.
4. Move each member to the *same quantile* of the new distribution:
   $x_n' = F_a^{-1}(q_n)$.

If $F_p$ and $F_a$ respect a bound, so does every updated member. The
**bounded normal rank histogram (BNRH)** distribution is the
near-universal non-parametric choice: by construction its quantiles are
exactly uniform. The cell below fits one and verifies the round trip.

In [ ]:
from pydartlab.algorithms.distributions import bnrh_fit
from scipy.stats import norm

ens = np.array([0.1, 0.3, 0.5, 0.8, 1.2])
dist = bnrh_fit(ens, bounded_below=True, lower_bound=0.0)
print("quantiles:", np.round(dist.sorted_quantiles, 3), "(uniform by construction)")
print("round trip ppf(cdf(x)) == x:",
      np.allclose(dist.ppf(dist.sorted_quantiles), dist.sort_x))

# probit transform: quantiles -> standard normal space
probit = norm.ppf(dist.sorted_quantiles)
print("probit-space ensemble:", np.round(probit, 3))

## Probit-space regression (PPI)

Section 2 regressed increments *linearly* in physical space, which can
push bounded unobserved variables out of bounds. The QCEFF fix: transform
**both** variables to standard normal space first,

$$ z = \Phi^{-1}\!\big(F_p(x)\big) \quad \text{(probit probability integral transform)}, $$

do the linear regression there (where everything is as Gaussian as it can
be), and transform back through $F_a^{-1}(\Phi(z))$. Bounds and
nonlinear relationships survive.

## Exercise: `twod_ppi_ensemble`

The unobserved (vertical) variable is non-negative. Compare distribution
choices:

1. **Normal/Normal** is exactly Section 2's regression — *challenge: make
   it produce negative posterior members* (strong correlation, members
   hugging zero, observation pulling down).
2. Switch the unobserved distribution to **Gamma** or **BNRH (bounded)**:
   the same update now respects the bound. Watch the right panel — the
   regression happens in the transformed space.
3. Try **RHF** for the observed variable with a bimodal observed prior.

In [ ]:
tp = apps.twod_ppi_ensemble()
tp.set_ensemble([(2.0, 0.1), (3.0, 0.4), (4.0, 0.9), (5.0, 1.6),
                 (6.0, 2.6), (7.0, 3.8), (8.0, 5.2)])
tp.obs_mean.value = 1.0   # observation pulls the observed variable down
tp.update_ensemble()      # Normal/Normal first
tp

In [ ]:
for sd in ("Normal", "Gamma", "BNRH (bounded)"):
    tp.state_dist.value = sd
    tp.update_ensemble()
    ymin = tp.last_posterior[:, 1].min()
    print(f"{sd:15s} min posterior unobserved value: {ymin: .4f}")

## Inflation in transformed space

Linear inflation can also push members past a bound. Inflating *in probit
space* (transform, inflate, transform back) cannot.

## Exercise: inflation in `bounded_oned_ensemble`

Enable inflation with a large value (3-5) for each filter choice. EAKF
inflates linearly (members can go negative); Gamma and BNRHF inflate in
their transformed spaces (members stay non-negative).

In [ ]:
bo2 = apps.bounded_oned_ensemble()
bo2.set_ensemble([0.05, 0.2, 0.5, 0.9, 1.5])
bo2.inf_toggle.value = True
bo2.inf_slider.value = 4.0
for f in ("EAKF", "BNRHF"):
    bo2.filter_radio.value = f
    bo2.update_ensemble()
    print(f"{f:6s} inflated prior min: {bo2.last_inflated_prior.min(): .4f}")
bo2

## What you should have seen

* An EAKF happily creates negative concentrations; gamma and BNRH filters
  cannot.
* Quantile conservation is the unifying trick: update the distribution,
  keep the quantiles.
* PPI/probit transforms extend the same idea to multivariate regression
  and to inflation.

These methods are available in the real DART system as **QCEFF tables**
(Section 6). **Next: Section 5 — letting the filter tune its own
inflation.**